# Imports

In [ ]:
import pandas as pd
import sys
import os

sys.path.append(os.path.abspath('../src'))
from A_TriggerChecks.extract_vmrk_path import extract_vmrk_path
from A_TriggerChecks.plot_trigger_counts import plot_trigger_counts
from A_TriggerChecks.compare_sequences import compare_sequences
from A_TriggerChecks.load_vmrk import load_vmrk
from A_TriggerChecks.load_trigger_map import load_trigger_map
from A_TriggerChecks.keep_rows_after_starts import keep_rows_after_starts
from A_TriggerChecks.repair_vmrk_D7_triggers import repair_vmrk_D7_triggers

# Variables

In [ ]:
subjectID = 'Pilote002'
task      = 'Task1_PPS'
data_path = '/Users/floremunier/Library/CloudStorage/Dropbox-NeuroRestore/Flore Munier-Jolain/PercepPD/Data'
csv_name1 = 'franc_s1_markers.csv'
csv_name2 = 'sub-franc_session-1_task1_trials.csv' 
repair_d7 = True

### Load Data

In [ ]:
# Extract .vmrk file path
vmrk2csv_save_path = os.path.join(data_path, subjectID, task, 'Output', 'Triggers', 'vmrk_triggers_' + task + '_' + subjectID + '.csv')
vmrk_path =  extract_vmrk_path(data_path, subjectID, task)
df_vmrk_markers = load_vmrk(vmrk_path, vmrk2csv_save_path)

# Extrat trigger value from file
xlsx_path = os.path.join(os.path.dirname(data_path),'Code/EEG_analysis/Scripts/src/Trigger_Values.xlsx')
trigger_map, _ = load_trigger_map(xlsx_path, sheet=task)

# Extrat csv log file with marker saved in unity
csv_path = vmrk_path =  os.path.join(data_path, subjectID, task, 'Raw', 'TaskData', csv_name1)
df_csv_markers = pd.read_csv(csv_path)
df_csv_markers["TriggerValue"] = pd.to_numeric(df_csv_markers["TriggerValue"], errors="coerce").astype("Int64")
df_csv_markers = df_csv_markers.sort_values("Time").reset_index(drop=True)

### EEG triggers count

In [ ]:
fig, counts = plot_trigger_counts(df_vmrk_markers, trigger_map, save_path=None, log_scale=False, title="Trigger occurrence count (EEG recording)")

### Triggers missmatch

In [ ]:
mismatches_save_path = os.path.join(data_path, subjectID, task, 'Output', 'Triggers', 'trigger_mismatches_' + task + '_' + subjectID + '.csv')
if task == 'Task1_PPS':
    norm_map={127: 63}
else:
    norm_map=None

mismatches_df, alignment_df = compare_sequences(df_vmrk_markers, df_csv_markers, mismatches_save_path, trigger_map=trigger_map, normalize_map=norm_map)


if repair_d7 and task == 'Task1_PPS':
    repaired_vmrk_save_path = os.path.join(data_path, subjectID, 'Task1_PPS', 'Output', 'Triggers', 'repaired_vmrk_triggers_task1_' + subjectID + '.csv')
    df_repaired_vmrk_markers = repair_vmrk_D7_triggers(vmrk2csv_save_path, mismatches_save_path, repaired_vmrk_save_path, trigger_map)
    fig, counts = plot_trigger_counts(df_repaired_vmrk_markers, trigger_map, save_path=None, log_scale=False, title="Trigger occurrence count (EEG recording)")

if task=='Task2_HitOrMiss':
    # Remove start/test triggers that are not useful for the analysis
    phase_block_value = 25 # TO UPDATE with dictonary
    df_vmrk_markers_t2 = keep_rows_after_starts(vmrk2csv_save_path, phase_block_value, vmrk2csv_save_path)
